# **Subagent**
**Handing off tasks to subagents isolates context, keeping the main (supervisor) agent’s context window clean while still going deep on a task.**
**The subagents middleware from `Deep Agents` allows you to supply subagents through a `task` tool.**

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [ ]:
## Imports
from deepagents.middleware.subagents import SubAgentMiddleware
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain_groq import ChatGroq 
from langchain.tools import tool


## Initialize the LLM
llm = ChatGroq(model="qwen/qwen3-32b")

## Define some tools
@tool
def read_file(path: str):
    """Read a file and return its contents."""
    with open(path, "r") as f:
        return f.read()
    
@tool
def write_file(path: str, content: str):
    """Write content to a file."""
    with open(path, "w") as f:
        f.write(content)

@tool
def make_directory(path: str):
    """Create directory if it doesn't exist."""
    os.makedirs(path, exist_ok=True)
    return f"Created {path}"

@tool
def create_file(path: str, content: str):
    """Create a file with the given content."""
    with open(path, "w") as f:
        f.write(content)
    return f"Created {path}"

@tool
def delete_file(path: str):
    """Delete a file."""
    os.remove(path)
    return f"Deleted {path}"

# Create a agent with subagent
agent = create_agent(
    model=llm,
    tools=[],
    middleware=[
        SubAgentMiddleware(
            default_model=llm,
            subagents=[
                {
                    "name": "file_manager",
                    "description": "Handles file operations like creating, reading, writing, and deleting files.",
                    "tools": [read_file, write_file, make_directory, create_file, delete_file],
                    "model" : llm
                }
            ]
        )
        ],
)